# Notebook 09 — Modèle final et soumission

## Objectif
Comparer les expériences initiales, sélectionner un premier modèle final, l'entraîner sur toutes les données et générer une première soumission. Les notebooks 10 à 15 poursuivent ensuite l'amélioration du modèle.

## Fichiers utilisés
- `data/X_train.csv`, `data/y_train.csv`, `data/X_test.csv`

## Résultat attendu
- Tableau comparatif de toutes les expériences
- Fichier `submissions/final.csv`
- Rapport Skore Hub (clé `07_final`)

## Notebook suivant
`10_improved_model.ipynb` — amélioration du modèle par tuning des hyperparamètres.

## 1. Imports

In [1]:
import json
import os
from pathlib import Path

import pandas as pd
import skore
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import GroupKFold

## 2. Connexion à Skore Hub

In [2]:
config = json.loads(Path("../.skore").read_text())
os.environ["SKORE_HUB_API_KEY"] = config["api_key"]
skore.login()
hub_project = skore.Project(
    "bobathon-esilv",
    mode="hub",
    workspace=config["workspace"],
)
print(f"Connecté au Hub : workspace={hub_project.workspace}")

╭───────────────────────────────── Login to Skore Hub ─────────────────────────────────╮
│                                                                                      │
│                        Successfully logged in, using API key.                        │
│                                                                                      │
╰──────────────────────────────────────────────────────────────────────────────────────╯

Connecté au Hub : workspace=BERNIE MADOFF


## 3. Chargement des données

In [3]:
DATA_DIR = Path("../data")

X_train_raw = pd.read_csv(DATA_DIR / "X_train.csv")
y_train     = pd.read_csv(DATA_DIR / "y_train.csv")
X_test_raw  = pd.read_csv(DATA_DIR / "X_test.csv")

visits = X_train_raw.merge(
    y_train,
    on="Index",
    how="inner",
    validate="one_to_one",
)

assert len(visits) == len(X_train_raw)
assert visits["target"].notna().all()

print(f"visits : {visits.shape}  |  X_test : {X_test_raw.shape}")

visits : (44590, 14)  |  X_test : (11013, 13)


## 4. Fonction de feature engineering

Les mêmes transformations sont appliquées à `visits` (train) et à `X_test_raw` pour éviter toute fuite.

In [4]:
def add_features(df: pd.DataFrame) -> pd.DataFrame:
    """Ajoute les features calculées. Applicable au train et au test."""
    df = df.copy()
    df["years_since_diagnosis"] = df["age"] - df["age_at_diagnosis"]
    for col in ["on", "off", "ledd", "time_since_intake_on", "time_since_intake_off"]:
        df[f"{col}_missing"] = df[col].isna().astype(int)
    df["off_on_gap"] = df["off"] - df["on"]
    return df

FEATURE_COLS = [
    "sexM", "age_at_diagnosis", "age", "ledd",
    "time_since_intake_on", "time_since_intake_off", "on", "off",
    "years_since_diagnosis",
    "on_missing", "off_missing", "ledd_missing",
    "time_since_intake_on_missing", "time_since_intake_off_missing",
    "off_on_gap",
]

visits_fe = add_features(visits)
X_test_fe = add_features(X_test_raw)

print(f"Features : {len(FEATURE_COLS)} colonnes")

Features : 15 colonnes


## 5. Tableau de comparaison des expériences

On recalcule les RMSE GroupKFold pour les configurations clés afin de construire le tableau de bord final.

In [5]:
from sklearn.dummy import DummyRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline

y = visits_fe["target"]
groups = visits_fe["patient_id"]

X_num = visits_fe[[
    "sexM", "age_at_diagnosis", "age", "ledd",
    "time_since_intake_on", "time_since_intake_off", "on", "off",
]]
X_fe = visits_fe[FEATURE_COLS]

cv_num = list(GroupKFold(n_splits=5).split(X_num, y, groups=groups))
cv_fe  = list(GroupKFold(n_splits=5).split(X_fe,  y, groups=groups))

experiments = [
    ("Dummy",     DummyRegressor(strategy="mean"),                    X_num, cv_num),
    ("Ridge",     make_pipeline(SimpleImputer(strategy="median"),
                                Ridge(alpha=0.1)),                    X_num, cv_num),
    ("HGBR num",  HistGradientBoostingRegressor(random_state=0),     X_num, cv_num),
    ("HGBR + FE", HistGradientBoostingRegressor(random_state=0),     X_fe,  cv_fe),
]

rows = []
for name, model, X, cv in experiments:
    rep = skore.evaluate(model, X, y, splitter=cv)
    rmse_vals = rep.metrics.rmse().iloc[:, 0].values
    rows.append({
        "Modèle": name,
        "RMSE moy.": rmse_vals.mean(),
        "RMSE std": rmse_vals.std(),
    })
    print(f"{name:15s}  RMSE={rmse_vals.mean():.4f} ± {rmse_vals.std():.4f}")

comparison_df = pd.DataFrame(rows).sort_values("RMSE moy.")
print()
display(comparison_df.reset_index(drop=True).round(4))

Output()

Output()

Dummy            RMSE=16.4992 ± 0.0000


Output()

Ridge            RMSE=10.5965 ± 0.0000


Output()

HGBR + FE        RMSE=7.6594 ± 0.0000



,Modèle,RMSE moy.,RMSE std
0,HGBR + FE,7.6594,0.0
1,HGBR num,7.7520,0.0
2,Ridge,10.5965,0.0
3,Dummy,16.4992,0.0


## 6. Sélection du modèle final

Le modèle sélectionné est celui avec la RMSE GroupKFold la plus basse et stable. On l'entraîne sur **toutes** les données d'entraînement.

In [6]:
final_model = HistGradientBoostingRegressor(random_state=0)
final_X_train = visits_fe[FEATURE_COLS]
final_y_train = visits_fe["target"]

final_model.fit(final_X_train, final_y_train)
print("Modèle final entraîné sur toutes les données")
print(f"Nb lignes d'entraînement : {len(final_X_train)}")

Modèle final entraîné sur toutes les données
Nb lignes d'entraînement : 44590


## 7. Prédictions sur le jeu de test

In [7]:
X_test_model = X_test_fe[FEATURE_COLS]
predictions = final_model.predict(X_test_model)

assert len(predictions) == len(X_test_raw)
assert pd.Series(predictions).notna().all()

print(f"Prédictions : {len(predictions)} lignes — OK")

Prédictions : 11013 lignes — OK


## 8. Construction et vérification du fichier de soumission

In [8]:
submission = X_test_raw[["Index"]].copy()
submission["target"] = predictions

assert list(submission.columns) == ["Index", "target"]
assert len(submission) == len(X_test_raw)
assert submission["Index"].equals(X_test_raw["Index"])
assert submission["target"].notna().all()

print("Vérifications soumission : OK")
display(submission.head())

Vérifications soumission : OK


,Index,target
0,66,26.892014
1,67,28.545493
2,68,35.063731
3,69,41.167247
4,70,32.584836


## 9. Enregistrement de la soumission

In [9]:
submissions_dir = Path("../submissions")
submissions_dir.mkdir(exist_ok=True)

submission.to_csv(submissions_dir / "final.csv", index=False)
print(f"Fichier enregistré : {submissions_dir / 'final.csv'}")

Fichier enregistré : ..\submissions\final.csv


## 10. Rapport Skore Hub du modèle final

In [10]:
report_final = skore.evaluate(
    HistGradientBoostingRegressor(random_state=0),
    final_X_train,
    final_y_train,
    splitter=cv_fe,
)

hub_project.put("07_final", report_final)
print("Rapport envoyé à Skore Hub sous la clé : 07_final")

Output()

Output()

Consult your report at https://skore.probabl.ai/BERNIE 
MADOFF/bobathon-esilv/cross-validations/61404

Rapport envoyé à Skore Hub sous la clé : 07_final


## 11. Conclusion et récapitulatif final

**Modèle retenu** : `HistGradientBoostingRegressor(random_state=0)` avec feature engineering (15 variables).

**Pourquoi ce choix :**
- Meilleure RMSE GroupKFold parmi toutes les configurations testées.
- Gestion native des valeurs manquantes — pas d'imputation arbitraire.
- Feature engineering stable : chaque variable améliore ou n'empire pas les performances.
- Pipeline reproductible : même transformation appliquée au train et au test.

**Limites :**
- La validation GroupKFold ne couvre que les patients présents dans le train. Les patients du test peuvent avoir des profils non représentés.
- `rater_id` n'a pas été inclus pour éviter un biais évaluateur.
- Le feature engineering est minimal : des interactions non linéaires supplémentaires pourraient encore améliorer le modèle.

**Fichier de soumission** : `submissions/final.csv`  
**Rapport Skore Hub** : clé `07_final`